In [14]:
import spacy
import pandas as pd
nlp = spacy.load("en_core_web_md")

In [15]:
df = pd.read_json("/natural-language-processing-models/spacy_word_vectors/news_dataset.json")
df.head()

,text,category
0,"Larry Nassar Blames His Victims, Says He 'Was ...",CRIME
1,"Woman Beats Cancer, Dies Falling From Horse",CRIME
2,Vegas Taxpayers Could Spend A Record $750 Mill...,SPORTS
3,This Richard Sherman Interception Literally Sh...,SPORTS
4,7 Things That Could Totally Kill Weed Legaliza...,BUSINESS


In [16]:
df['category_label_num'] = df['category'].map({'CRIME':0,'BUSINESS':1,'SPORTS':2})
df.head()

,text,category,category_label_num
0,"Larry Nassar Blames His Victims, Says He 'Was ...",CRIME,0
1,"Woman Beats Cancer, Dies Falling From Horse",CRIME,0
2,Vegas Taxpayers Could Spend A Record $750 Mill...,SPORTS,2
3,This Richard Sherman Interception Literally Sh...,SPORTS,2
4,7 Things That Could Totally Kill Weed Legaliza...,BUSINESS,1


In [17]:
def preprocess(text):
    doc = nlp(text)
    filtered_sentence = []
    for word in doc:
        if word.is_punct or word.is_stop:
            continue
        else:
            filtered_sentence.append(word.lemma_)
    return " ".join(filtered_sentence)

In [18]:
df['filtered_text'] = df['text'].apply(lambda text : preprocess(text))
df.head()

,text,category,category_label_num,filtered_text
0,"Larry Nassar Blames His Victims, Says He 'Was ...",CRIME,0,Larry Nassar blame victim say victimize Newly ...
1,"Woman Beats Cancer, Dies Falling From Horse",CRIME,0,Woman Beats Cancer dies fall horse
2,Vegas Taxpayers Could Spend A Record $750 Mill...,SPORTS,2,Vegas Taxpayers spend Record $ 750 million New...
3,This Richard Sherman Interception Literally Sh...,SPORTS,2,Richard Sherman Interception literally shake W...
4,7 Things That Could Totally Kill Weed Legaliza...,BUSINESS,1,7 thing totally kill Weed Legalization Buzz


In [19]:
#spacy glove embeddings for text vectorization.

df['word_vector'] = df['filtered_text'].apply(lambda text : nlp(text).vector)
df.head()

,text,category,category_label_num,filtered_text,word_vector
0,"Larry Nassar Blames His Victims, Says He 'Was ...",CRIME,0,Larry Nassar blame victim say victimize Newly ...,"[-0.7315932, -0.0041999375, -0.13447016, 0.001..."
1,"Woman Beats Cancer, Dies Falling From Horse",CRIME,0,Woman Beats Cancer dies fall horse,"[-0.68895173, 0.19709416, -0.09449599, 0.03946..."
2,Vegas Taxpayers Could Spend A Record $750 Mill...,SPORTS,2,Vegas Taxpayers spend Record $ 750 million New...,"[-0.6947, 0.060308475, 0.04005947, -0.125657, ..."
3,This Richard Sherman Interception Literally Sh...,SPORTS,2,Richard Sherman Interception literally shake W...,"[-0.680972, 0.18936579, 0.12709197, -0.2889879..."
4,7 Things That Could Totally Kill Weed Legaliza...,BUSINESS,1,7 thing totally kill Weed Legalization Buzz,"[-0.7347385, 0.22293714, -0.08762856, -0.18832..."


In [20]:
from sklearn.model_selection import train_test_split
import numpy as np
x_train,x_test,y_train,y_test = train_test_split(df['word_vector'],df['category_label_num'],test_size=0.2)

print(x_train.shape, x_test.shape)

x_train_2d = np.stack(x_train)
x_test_2d = np.stack(x_test)

print(x_train_2d.shape, x_test_2d.shape)


(6000,) (1500,)
(6000, 300) (1500, 300)


In [30]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

classifier = DecisionTreeClassifier()
classifier.fit(x_train_2d,y_train)
y_pred = classifier.predict(x_test_2d)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.70      0.70      0.70       476
           1       0.74      0.73      0.74       525
           2       0.70      0.70      0.70       499

    accuracy                           0.71      1500
   macro avg       0.71      0.71      0.71      1500
weighted avg       0.71      0.71      0.71      1500



In [31]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler

classifier = MultinomialNB()
scaler = MinMaxScaler()

sca_train_embed = scaler.fit_transform(x_train_2d)
sca_test_embed = scaler.transform(x_test_2d)

classifier.fit(sca_train_embed,y_train)
y_pred = classifier.predict(sca_test_embed)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.84      0.81      0.82       476
           1       0.82      0.84      0.83       525
           2       0.79      0.80      0.79       499

    accuracy                           0.82      1500
   macro avg       0.82      0.81      0.82      1500
weighted avg       0.82      0.82      0.82      1500



In [32]:
from sklearn.neighbors import KNeighborsClassifier

classifier = KNeighborsClassifier(metric = 'euclidean')

classifier.fit(x_train_2d,y_train)
y_pred = classifier.predict(x_test_2d)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.79      0.89      0.84       476
           1       0.84      0.87      0.85       525
           2       0.90      0.75      0.82       499

    accuracy                           0.84      1500
   macro avg       0.84      0.84      0.84      1500
weighted avg       0.84      0.84      0.84      1500



In [33]:
from sklearn.ensemble import RandomForestClassifier

classifier = RandomForestClassifier()

classifier.fit(x_train_2d,y_train)
y_pred = classifier.predict(x_test_2d)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.83      0.85      0.84       476
           1       0.86      0.84      0.85       525
           2       0.82      0.82      0.82       499

    accuracy                           0.84      1500
   macro avg       0.84      0.84      0.84      1500
weighted avg       0.84      0.84      0.84      1500



In [34]:
from sklearn.ensemble import GradientBoostingClassifier


classifier = GradientBoostingClassifier()

classifier.fit(x_train_2d,y_train)
y_pred = classifier.predict(x_test_2d)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.86      0.88      0.87       476
           1       0.88      0.87      0.88       525
           2       0.87      0.87      0.87       499

    accuracy                           0.87      1500
   macro avg       0.87      0.87      0.87      1500
weighted avg       0.87      0.87      0.87      1500

